# 🎵 Pilotage — Catalogue des chants OpenSong

**Commencer ici.** Ce notebook fait tourner tout le projet, cellule par cellule :
clique dans une cellule puis **Maj + Entrée** pour l'exécuter et passer à la suivante.

| Partie | Ce qu'elle fait |
|---|---|
| 1. Lire | lit et nettoie tous les fichiers OpenSong |
| 2. Comparer | trouve les chants qui se ressemblent et leur donne un score |
| 3. Cataloguer | regroupe en **chants** (CH-…) avec leurs **versions** (complète, courte, traduction…) |
| 4. Chercher | retrouve un chant malgré les fautes et affiche sa fiche |
| 5. Préparer un culte | colle une liste de chants → set OpenSong prêt à projeter |
| 6-7. Annoter / évaluer | vérifie à la main un échantillon pour régler la machine |

Le dossier OpenSong est seulement **lu**, jamais modifié. Les calculs sont faits par le code
du dossier `fondary/` : ce notebook ne fait que l'appeler et afficher les résultats.

## 0. Préparation

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sqlite3
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, HTML

# Se placer à la racine du projet (le dossier qui contient fondary/)
ROOT = Path.cwd()
while not (ROOT / "fondary").is_dir():
    ROOT = ROOT.parent
os.chdir(ROOT)

SOURCE = Path("chant_xml/Chants XML")   # ← dossier OpenSong à analyser
DB = Path("data/catalogue.sqlite")
REPORTS = Path("reports")

pd.set_option("display.max_colwidth", 80)
print("Projet :", ROOT)
print("Dossier OpenSong :", SOURCE, "→", "trouvé ✔" if SOURCE.is_dir() else "INTROUVABLE ✘")

## 1. Lire et nettoyer les fichiers OpenSong

Lit chaque fichier, découpe les paroles en sections (couplets, refrain…), prépare les clés de
comparaison (sans accents ni ponctuation), détecte la langue. Durée : ~30 secondes.

In [ ]:
from fondary.extract import extract, save_sqlite
from fondary.stats import build_report, write_outputs

files, sections = extract(SOURCE)
save_sqlite(files, sections, DB)
report, sheets = build_report(files, sections, SOURCE)
write_outputs(report, sheets, REPORTS)
print(f"{len(files)} fichiers lus, {len(sections)} sections")

Le rapport de qualité complet (détails dans `reports/rapport_extraction.xlsx`) :

In [ ]:
display(Markdown(report))

## 2. Comparer

1. Les fichiers aux paroles **identiques** sont regroupés en une même *version*.
2. Le programme cherche les paires de versions qui se ressemblent et leur donne un **score** de 0 à 1.
3. Chaque paire reçoit une **bande** (très forte, probable, incertaine, rejet) et une **relation proposée**.

In [ ]:
from fondary.catalog import build_versions
from fondary.matching import run_matching
from fondary.match_report import build_match_report

con = sqlite3.connect(DB)
versions = build_versions(con)
paires = run_matching(con)
n_files = con.execute("SELECT COUNT(*) FROM file_version").fetchone()[0]
(REPORTS / "rapport_matching.md").write_text(build_match_report(versions, paires, n_files), encoding="utf-8")

print(f"{n_files} fichiers → {len(versions)} textes distincts ; {len(paires)} paires comparées")
ordre = ["tres_forte", "probable", "incertaine", "rejet"]
display(pd.crosstab(paires.confidence_band, paires.predicted_relation, margins=True, margins_name="total")
          .reindex(ordre + ["total"]))

## 3. Cataloguer : chants et versions

- Les textes qui parlent du **même chant** sont regroupés sous un identifiant **CH-xxxxx** qui ne change plus.
- Un garde-fou évite les chaînes (A ressemble à B, B à C, mais A pas à C).
- Les quasi-doublons (fautes, ponctuation) sont fusionnés dans une même version.
- Chaque version reçoit un **code** stable (`CH-00218.fr.courte`) et un **libellé** lisible
  (« FR · courte — couplets 1 et 3 + refrain »), décrit par rapport à la version de référence.

In [ ]:
from fondary.canonical import build_catalogue, catalogue_report

chants, versions_cat, infos = build_catalogue(con)
display(Markdown(catalogue_report(chants, versions_cat, infos)))

## 4. Chercher un chant

Tape un titre, une première ligne ou une phrase entendue — les fautes et accents sont tolérés.

In [ ]:
from fondary.search import build_index, search, song_card, resolve_list, export_set

index = build_index(con)

def fiche(song_id):
    """Affiche la fiche d'un chant : ses versions (référence en tête) et le fichier à ouvrir."""
    chant, vers = song_card(con, song_id)
    display(Markdown(f"### {chant.song_id} · {chant.canonical_title}\n"
                     f"Autres titres : {chant.alt_titles or '—'}  \n"
                     f"Auteurs : {chant.authors or '—'} · Recueils : {chant.collections or '—'}"))
    display(vers[["version_code", "version_label", "fichier_prefere", "n_files", "keys"]])
    return vers

def _texte(version_id):
    rows = con.execute("""SELECT s.label, s.text_display FROM song_version v
                          JOIN section s ON s.file_id = v.preferred_file_id
                          WHERE v.version_id = ? ORDER BY s.position""", (version_id,)).fetchall()
    code_v, label = con.execute("SELECT version_code, version_label FROM version_catalogue WHERE version_id=?",
                                (version_id,)).fetchone() or (version_id, "")
    corps = "".join(f"<p><b>[{l}]</b><br>{t.replace(chr(10), '<br>')}</p>" for l, t in rows)
    return f"<h4>{code_v}</h4><small>{label}</small>{corps}"

def comparer(a, b):
    """Affiche deux versions côte à côte (version_id)."""
    display(HTML(f"""<div style="display:flex;gap:24px">
        <div style="flex:1;border:1px solid #ccc;padding:8px">{_texte(a)}</div>
        <div style="flex:1;border:1px solid #ccc;padding:8px">{_texte(b)}</div></div>"""))

In [ ]:
resultats = search(index, "amazing grace")   # ← tape un titre ou une phrase
resultats

In [ ]:
vers = fiche(resultats.song_id.iloc[0])   # ← ou fiche("CH-00218")

In [ ]:
if len(vers) > 1:
    comparer(vers.version_id.iloc[0], vers.version_id.iloc[1])   # référence vs 2e version
else:
    print("Ce chant n'a qu'une version.")

## 5. Préparer un culte : liste → set OpenSong

Colle la liste reçue (un chant par ligne, numéros et tirets acceptés).
Chaque ligne est reliée à un chant et à sa version de référence :

- **ok** : sûr · **à vérifier** : plusieurs chants proches, regarde `autres_candidats` · **introuvable** : exclu du set.

Le set est écrit dans `exports/sets/` : copie-le dans le dossier **Sets** d'OpenSong.

In [ ]:
LISTE = """
1. A toi la gloire
2. Amazing grace
3. Comme un cerf altéré
4. Ce feu en moi
"""
NOM_DU_SET = "Culte 2026-11-15"

resolution = resolve_list(con, index, LISTE)
display(resolution)

In [ ]:
chemin = export_set(resolution, NOM_DU_SET, Path("exports/sets"))
print("Set écrit :", chemin.resolve())

## 6. Annoter des paires (pour régler la machine)

But : dire, pour ~300 paires, si la machine a raison.

1. La cellule suivante crée `annotations/paires_a_annoter.xlsx` (si elle n'existe pas déjà).
2. Ouvre-le dans Excel, remplis la colonne **label_humain** (liste déroulante), enregistre, **ferme Excel**.
3. Lance la cellule d'import, puis relance les parties **2 et 3** : tes réponses sont prises en compte.

In [ ]:
from fondary.annotate import export_sample, import_annotations

fichier_annot = Path("annotations/paires_a_annoter.xlsx")
if fichier_annot.exists():
    print("Le fichier existe déjà :", fichier_annot.resolve())
else:
    print(export_sample(con, fichier_annot), "paires exportées →", fichier_annot.resolve())

In [ ]:
ANNOTATEUR = "Kiswendsida"   # ← ton nom
ok, ignorees = import_annotations(con, fichier_annot, ANNOTATEUR)
print(f"{ok} annotations enregistrées, {ignorees} ignorées (« ? » ou vides)")

## 7. Évaluation

Après import des annotations et relance de la partie 2 :

In [ ]:
ann = paires.dropna(subset=["human_relation"])
if ann.empty:
    print("Pas encore d'annotations importées.")
else:
    display(Markdown((REPORTS / "rapport_matching.md").read_text(encoding="utf-8").split("## Évaluation")[-1]))